# Animal Detector Trainer
This notebook takes videos, makes labeled image sets, trains a YOLO detector, and shows where the animal is in each frame.

## Step 1: Setup
1. Point to the folder with your videos.
2. Pick how many frames to save from each video.
3. Run the next cell to install and load the tools.

In [1]:
# Install dependencies (run once)
%pip -q install ultralytics opencv-python torch torchvision

# Imports
from pathlib import Path
import random
import shutil
import yaml
import csv
import numpy as np
from ultralytics import YOLO

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Note: you may need to restart the kernel to use updated packages.
Using device: cuda


In [2]:
# Editable settings
video_folder = Path("/media/crow/My Passport/BR Footage/all")
frames_per_video = 25
image_file_extension = ".jpg"

shuffle_seed = 812
train_fraction = 0.8

# Output dataset folders
dataset_root = Path("/media/crow/My Passport/BR Footage/Gharial/target/validate/dataset")
train_images_dir = dataset_root / "images" / "train"
validation_images_dir = dataset_root / "images" / "val"
train_labels_dir = dataset_root / "labels" / "train"
validation_labels_dir = dataset_root / "labels" / "val"

if frames_per_video <= 0:
    raise ValueError(f"frames_per_video must be > 0, got {frames_per_video}")
if not (0 < train_fraction < 1):
    raise ValueError(f"train_fraction must be between 0 and 1, got {train_fraction}")
if image_file_extension.lower() not in {'.jpg', '.jpeg', '.png'}:
    raise ValueError("image_file_extension must be one of: .jpg, .jpeg, .png")

for folder in [train_images_dir, validation_images_dir, train_labels_dir, validation_labels_dir]:
    folder.mkdir(parents=True, exist_ok=True)

PermissionError: [Errno 13] Permission denied: '/media/crow/My Passport'

## Step 2: Extract frames from videos
1. Put your videos in the folder you set in Step 1.
2. Turn on `RUN_FRAME_EXTRACTION` in the next cell.
3. Frames will be saved into one folder per video.

In [3]:
RUN_FRAME_EXTRACTION = False
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv"}

def extract_evenly_spaced_frames(
    video_path: Path,
    frames_to_save: int,
    output_folder: Path,
    image_extension: str = ".jpg",
) -> int:
    import cv2

    if frames_to_save <= 0:
        raise ValueError("frames_to_save must be > 0")

    video_capture = cv2.VideoCapture(str(video_path))
    if not video_capture.isOpened():
        print(f"Skipping {video_path.name}: cannot open")
        return 0

    total_frame_count = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frame_count <= 0:
        print(f"Skipping {video_path.name}: no frames")
        video_capture.release()
        return 0

    output_folder.mkdir(parents=True, exist_ok=True)
    frame_indices = np.linspace(0, total_frame_count - 1, frames_to_save, dtype=int)

    saved_count = 0
    for frame_index in frame_indices:
        video_capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = video_capture.read()
        if not ok or frame is None:
            continue

        output_path = output_folder / f"{video_path.stem}_f{frame_index:06d}{image_extension}"
        if cv2.imwrite(str(output_path), frame):
            saved_count += 1

    video_capture.release()
    return saved_count

if RUN_FRAME_EXTRACTION:
    if not video_folder.exists() or not video_folder.is_dir():
        raise FileNotFoundError(f"video_folder does not exist or is not a directory: {video_folder}")

    video_paths = sorted([path for path in video_folder.iterdir() if path.suffix.lower() in VIDEO_EXTENSIONS])
    if not video_paths:
        print("No videos found. Check video_folder.")
    else:
        for video_path in video_paths:
            output_folder = video_path.parent / f"{video_path.stem}_frames"
            saved = extract_evenly_spaced_frames(
                video_path,
                frames_per_video,
                output_folder,
                image_file_extension,
            )
            print(f"{video_path.name}: saved {saved} frames to {output_folder}")

## Step 3: Build train and validation folders
1. Collect labeled images and YOLO label files in one place.
2. Point `labeled_data_root` to that folder.
3. The cell splits images into train/val and writes `data.yaml`.

In [5]:
# Folder that contains images and YOLO .txt labels with matching names
labeled_data_root = Path("/media/crow/My Passport/BR Footage/Gharial/target/train_compiled")

# Class names in your dataset (order must match label IDs)
class_labels = ["gharial"]

# If an image has no labels, create an empty .txt so it can be used as a negative example
include_unlabeled_images = True

IMAGE_FILE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

if not labeled_data_root.exists() or not labeled_data_root.is_dir():
    raise FileNotFoundError(f"labeled_data_root does not exist or is not a directory: {labeled_data_root}")
if not class_labels:
    raise ValueError("class_labels must contain at least one class name.")


def collect_image_label_pairs(
    root: Path,
    extensions=IMAGE_FILE_EXTENSIONS,
    include_unlabeled: bool = True,
):
    image_label_pairs = []
    images_dir = root / "images"
    labels_dir = root / "labels"

    if images_dir.exists() and labels_dir.exists():
        label_files_by_stem = {}
        for label_file in labels_dir.rglob("*.txt"):
            label_files_by_stem[label_file.stem] = label_file

        for image_path in images_dir.rglob("*"):
            if image_path.suffix.lower() in extensions:
                label_path = label_files_by_stem.get(image_path.stem)
                if label_path and label_path.exists():
                    image_label_pairs.append((image_path, label_path))
                elif include_unlabeled:
                    image_label_pairs.append((image_path, None))
        return image_label_pairs

    for image_path in root.rglob("*"):
        if image_path.suffix.lower() in extensions:
            label_path = image_path.with_suffix(".txt")
            if label_path.exists():
                image_label_pairs.append((image_path, label_path))
            elif include_unlabeled:
                image_label_pairs.append((image_path, None))

    return image_label_pairs


image_label_pairs = collect_image_label_pairs(
    labeled_data_root,
    include_unlabeled=include_unlabeled_images,
)
if not image_label_pairs:
    image_count = 0
    for image_candidate in labeled_data_root.rglob("*"):
        if image_candidate.suffix.lower() in IMAGE_FILE_EXTENSIONS:
            image_count += 1

    label_count = 0
    for label_file in labeled_data_root.rglob("*.txt"):
        _ = label_file
        label_count += 1

    raise RuntimeError(
        f"No images found in {labeled_data_root}. "
        f"Found {image_count} images and {label_count} labels. "
        "Ensure the folder contains images (and optional .txt labels) or use images/ and labels/ subfolders."
    )

random.seed(shuffle_seed)
random.shuffle(image_label_pairs)
split_index = int(len(image_label_pairs) * train_fraction)
if split_index <= 0 or split_index >= len(image_label_pairs):
    raise RuntimeError(
        "Invalid train/val split produced an empty train or val set. "
        f"pairs={len(image_label_pairs)}, train_fraction={train_fraction}"
    )

train_pairs = image_label_pairs[:split_index]
validation_pairs = image_label_pairs[split_index:]


def copy_image_label_pairs(
    pairs_to_copy,
    destination_images_dir: Path,
    destination_labels_dir: Path,
):
    destination_images_dir.mkdir(parents=True, exist_ok=True)
    destination_labels_dir.mkdir(parents=True, exist_ok=True)
    for image_path, label_path in pairs_to_copy:
        shutil.copy2(image_path, destination_images_dir / image_path.name)
        output_label_path = destination_labels_dir / f"{image_path.stem}.txt"
        if label_path is None:
            output_label_path.write_text("", encoding="utf-8")
        else:
            shutil.copy2(label_path, output_label_path)


copy_image_label_pairs(train_pairs, train_images_dir, train_labels_dir)
copy_image_label_pairs(validation_pairs, validation_images_dir, validation_labels_dir)

dataset_config = {
    "path": str(dataset_root),
    "train": "images/train",
    "val": "images/val",
    "nc": len(class_labels),
    "names": class_labels,
}

dataset_root.mkdir(parents=True, exist_ok=True)
with open(dataset_root / "data.yaml", "w", encoding="utf-8") as file_handle:
    yaml.safe_dump(dataset_config, file_handle, sort_keys=False)

print(f"Train images: {len(train_pairs)} | Val images: {len(validation_pairs)}")
print(f"Wrote {dataset_root / 'data.yaml'}")

Train images: 998 | Val images: 250
Wrote /media/crow/My Passport/BR Footage/Gharial/target/validate/dataset/data.yaml


## Step 3b: Build classification dataset from YOLO labels
1. Crops bounding-box regions as positive samples and random background patches as negatives.
2. Used to fine-tune AlexNet and Inception classifiers alongside YOLO.
3. Images are resized to a common size for the classification models.

In [ ]:
# Classification dataset settings
classification_image_size = 224  # AlexNet & Inception input size
classification_root = dataset_root / "classification"
cls_train_pos = classification_root / "train" / "positive"
cls_train_neg = classification_root / "train" / "negative"
cls_val_pos = classification_root / "val" / "positive"
cls_val_neg = classification_root / "val" / "negative"

for d in [cls_train_pos, cls_train_neg, cls_val_pos, cls_val_neg]:
    d.mkdir(parents=True, exist_ok=True)


def build_classification_crops(
    pairs: list,
    pos_dir: Path,
    neg_dir: Path,
    crop_size: int = 224,
    max_negatives_per_image: int = 2,
):
    """Crop bounding-box regions (positives) and random patches (negatives) from YOLO-labelled images."""
    import cv2

    pos_count, neg_count = 0, 0
    for image_path, label_path in pairs:
        img = cv2.imread(str(image_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        boxes = []
        if label_path is not None and label_path.exists():
            for line in label_path.read_text(encoding="utf-8").strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                _, cx, cy, bw, bh = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                if x2 - x1 > 10 and y2 - y1 > 10:
                    boxes.append((x1, y1, x2, y2))

        # Save positive crops
        for (x1, y1, x2, y2) in boxes:
            crop = img[y1:y2, x1:x2]
            crop_resized = cv2.resize(crop, (crop_size, crop_size))
            out = pos_dir / f"{image_path.stem}_pos{pos_count}.jpg"
            cv2.imwrite(str(out), crop_resized)
            pos_count += 1

        # Save negative crops (random patches that don't overlap boxes)
        neg_saved = 0
        attempts = 0
        while neg_saved < max_negatives_per_image and attempts < 50:
            attempts += 1
            rx1 = random.randint(0, max(0, w - crop_size))
            ry1 = random.randint(0, max(0, h - crop_size))
            rx2 = min(rx1 + crop_size, w)
            ry2 = min(ry1 + crop_size, h)

            # Check overlap with any positive box
            overlaps = False
            for (bx1, by1, bx2, by2) in boxes:
                if rx1 < bx2 and rx2 > bx1 and ry1 < by2 and ry2 > by1:
                    overlaps = True
                    break
            if overlaps:
                continue

            crop = img[ry1:ry2, rx1:rx2]
            crop_resized = cv2.resize(crop, (crop_size, crop_size))
            out = neg_dir / f"{image_path.stem}_neg{neg_count}.jpg"
            cv2.imwrite(str(out), crop_resized)
            neg_count += 1
            neg_saved += 1

    return pos_count, neg_count


train_pos, train_neg = build_classification_crops(train_pairs, cls_train_pos, cls_train_neg, classification_image_size)
val_pos, val_neg = build_classification_crops(validation_pairs, cls_val_pos, cls_val_neg, classification_image_size)
print(f"Classification dataset — Train: {train_pos} pos / {train_neg} neg | Val: {val_pos} pos / {val_neg} neg")

## Step 4a: Train YOLO detector
1. Choose a base model or let the notebook pick the most recent run.
2. Train and validate on your detection dataset.
3. Best weights saved under `runs/detect/train_custom/weights/`.

In [ ]:
def find_most_recent_weights(search_dir: Path) -> Path | None:
    if not search_dir.exists():
        return None
    weight_files = list(search_dir.rglob("*/weights/best.pt")) + list(search_dir.rglob("*/weights/last.pt"))
    if not weight_files:
        return None
    return max(weight_files, key=lambda path: path.stat().st_mtime)


runs_detect_dir = Path("runs") / "detect"
latest_weights = find_most_recent_weights(runs_detect_dir)
starting_model = Path(latest_weights) if latest_weights else Path("yolov8s.pt")
if not starting_model.exists():
    raise FileNotFoundError(f"Starting model was not found: {starting_model}")

print(f"Using base model: {starting_model}")

try:
    yolo_model = YOLO(str(starting_model))
except Exception as exc:
    raise RuntimeError(f"Failed to initialize YOLO model from {starting_model}: {exc}") from exc


def load_training_history(results_csv_path: Path):
    if not results_csv_path.exists():
        return None
    with open(results_csv_path, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    if not rows:
        return None
    history = {key: [] for key in rows[0].keys()}
    for row in rows:
        for key, value in row.items():
            try:
                history[key].append(float(value))
            except Exception:
                history[key].append(value)
    return history


def plot_metric(history, axis, metric_key, label):
    if metric_key in history:
        epochs = history.get("epoch", list(range(len(history[metric_key]))))
        axis.plot(epochs, history[metric_key], label=label)
        return True
    return False


def plot_training_progress(history):
    clear_output(wait=True)
    _, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.flatten()

    axes[0].set_title("Box Loss")
    has_line = plot_metric(history, axes[0], "train/box_loss", "train")
    has_line = plot_metric(history, axes[0], "val/box_loss", "val") or has_line
    if has_line:
        axes[0].legend()

    axes[1].set_title("Class Loss")
    has_line = plot_metric(history, axes[1], "train/cls_loss", "train")
    has_line = plot_metric(history, axes[1], "val/cls_loss", "val") or has_line
    if has_line:
        axes[1].legend()

    axes[2].set_title("DFL Loss")
    has_line = plot_metric(history, axes[2], "train/dfl_loss", "train")
    has_line = plot_metric(history, axes[2], "val/dfl_loss", "val") or has_line
    if has_line:
        axes[2].legend()

    axes[3].set_title("Precision")
    if plot_metric(history, axes[3], "metrics/precision(B)", "precision"):
        axes[3].legend()

    axes[4].set_title("Recall")
    if plot_metric(history, axes[4], "metrics/recall(B)", "recall"):
        axes[4].legend()

    axes[5].set_title("mAP")
    has_line = plot_metric(history, axes[5], "metrics/mAP50(B)", "mAP50")
    has_line = plot_metric(history, axes[5], "metrics/mAP50-95(B)", "mAP50-95") or has_line
    if has_line:
        axes[5].legend()

    for axis in axes:
        axis.set_xlabel("epoch")
    plt.tight_layout()
    plt.show()


def update_training_plot(trainer):
    history = load_training_history(trainer.save_dir / "results.csv")
    if history is not None:
        plot_training_progress(history)


data_yaml_path = dataset_root / "data.yaml"
if not data_yaml_path.exists():
    raise FileNotFoundError(f"Dataset config not found: {data_yaml_path}. Run Step 3 first.")

yolo_model.add_callback("on_train_epoch_end", update_training_plot)

yolo_training_results = yolo_model.train(
    data=str(data_yaml_path),
    epochs=15,
    imgsz=800,
    batch=16,
    project="runs/detect",
    name="train_custom",
    exist_ok=True,
)

yolo_val_results = yolo_model.val(data=str(data_yaml_path))
print("YOLO validation:", yolo_val_results)

Using base model: runs/detect/train_custom/weights/last.pt
New https://pypi.org/project/ultralytics/8.4.14 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.12 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 5744MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/media/crow/My Passport/BR Footage/Gharial/target/validate/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio

RuntimeError: CUDA error: CUBLAS_STATUS_ALLOC_FAILED when calling `cublasCreate(handle)`

## Step 4b: Shared classifier training utilities
Reusable `Dataset`, training loop, and evaluation used by both AlexNet and Inception.

In [ ]:
# ── Classification dataset & training helpers ──────────────────────────

class CropDataset(Dataset):
    """Binary classification dataset: folders named 'positive' and 'negative'."""

    def __init__(self, root: Path, transform=None):
        self.samples: list[tuple[Path, int]] = []
        self.transform = transform
        pos_dir = root / "positive"
        neg_dir = root / "negative"
        if pos_dir.exists():
            for p in sorted(pos_dir.iterdir()):
                if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.samples.append((p, 1))
        if neg_dir.exists():
            for p in sorted(neg_dir.iterdir()):
                if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.samples.append((p, 0))
        if not self.samples:
            raise RuntimeError(f"No images found in {root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


# Transforms (ImageNet normalisation)
classifier_train_transform = transforms.Compose([
    transforms.Resize((classification_image_size, classification_image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

classifier_val_transform = transforms.Compose([
    transforms.Resize((classification_image_size, classification_image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# DataLoaders
classifier_batch_size = 32

train_cls_dataset = CropDataset(classification_root / "train", transform=classifier_train_transform)
val_cls_dataset = CropDataset(classification_root / "val", transform=classifier_val_transform)

train_cls_loader = DataLoader(train_cls_dataset, batch_size=classifier_batch_size, shuffle=True, num_workers=2)
val_cls_loader = DataLoader(val_cls_dataset, batch_size=classifier_batch_size, shuffle=False, num_workers=2)

print(f"Classification loaders — Train: {len(train_cls_dataset)} samples | Val: {len(val_cls_dataset)} samples")


# ── Generic training / evaluation loop ─────────────────────────────────

def train_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 15,
    lr: float = 1e-4,
    model_name: str = "classifier",
    save_dir: Path = Path("runs/classify"),
):
    """Fine-tune a binary classifier and save the best weights."""
    save_dir.mkdir(parents=True, exist_ok=True)
    best_path = save_dir / f"{model_name}_best.pt"

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    best_val_acc = 0.0
    history: dict[str, list] = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        # ── train ──
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            # Inception v3 returns InceptionOutputs in training mode
            if isinstance(outputs, tuple):
                outputs = outputs[0]
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        # ── validate ──
        model.eval()
        val_loss_sum, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                if isinstance(outputs, tuple):
                    outputs = outputs[0]
                val_loss_sum += criterion(outputs, labels).item() * images.size(0)
                correct += (outputs.argmax(dim=1) == labels).sum().item()
                total += labels.size(0)
        val_loss = val_loss_sum / total
        val_acc = correct / total
        scheduler.step(val_loss)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)

        print(f"[{model_name}] Epoch {epoch}/{epochs}  "
              f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history["epoch"], history["train_loss"], label="train")
    ax1.plot(history["epoch"], history["val_loss"], label="val")
    ax1.set_title(f"{model_name} — Loss")
    ax1.legend()
    ax2.plot(history["epoch"], history["val_acc"], label="val accuracy")
    ax2.set_title(f"{model_name} — Accuracy")
    ax2.legend()
    plt.tight_layout()
    plt.show()

    print(f"Best {model_name} val accuracy: {best_val_acc:.4f}  →  {best_path}")
    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
    return model, history

## Step 4c: Fine-tune AlexNet
Pre-trained AlexNet with its final classifier replaced for binary classification (target vs. background).

In [ ]:
classifier_epochs = 15
num_classes = 2  # positive / negative

# ── Build AlexNet ──────────────────────────────────────────────────────
alexnet_model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
alexnet_model.classifier[6] = nn.Linear(alexnet_model.classifier[6].in_features, num_classes)

alexnet_model, alexnet_history = train_classifier(
    alexnet_model,
    train_cls_loader,
    val_cls_loader,
    epochs=classifier_epochs,
    lr=1e-4,
    model_name="alexnet",
    save_dir=Path("runs/classify"),
)

## Step 4d: Fine-tune Inception v3
Pre-trained Inception v3 adapted for binary classification. Inception v3 expects 299×299 input, so a separate transform and loader are used.

In [ ]:
# Inception v3 requires 299×299 input
inception_image_size = 299

inception_train_transform = transforms.Compose([
    transforms.Resize((inception_image_size, inception_image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

inception_val_transform = transforms.Compose([
    transforms.Resize((inception_image_size, inception_image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

inception_train_dataset = CropDataset(classification_root / "train", transform=inception_train_transform)
inception_val_dataset = CropDataset(classification_root / "val", transform=inception_val_transform)

inception_train_loader = DataLoader(inception_train_dataset, batch_size=classifier_batch_size, shuffle=True, num_workers=2)
inception_val_loader = DataLoader(inception_val_dataset, batch_size=classifier_batch_size, shuffle=False, num_workers=2)

# ── Build Inception v3 ─────────────────────────────────────────────────
inception_model = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
inception_model.fc = nn.Linear(inception_model.fc.in_features, num_classes)
inception_model.aux_logits = True
inception_model.AuxLogits.fc = nn.Linear(inception_model.AuxLogits.fc.in_features, num_classes)

inception_model, inception_history = train_classifier(
    inception_model,
    inception_train_loader,
    inception_val_loader,
    epochs=classifier_epochs,
    lr=1e-4,
    model_name="inception",
    save_dir=Path("runs/classify"),
)

## Step 4e: Ensemble setup
Combines YOLO (detection) with AlexNet and Inception (classification) into a single ensemble predictor.

**Strategy:** YOLO proposes bounding boxes. Each crop is classified by AlexNet and Inception. A detection is kept only if the average positive-class probability across the classifiers exceeds `ensemble_threshold`.

In [ ]:
ensemble_threshold = 0.5  # minimum average positive-class probability to keep a detection

# ── Helpers ────────────────────────────────────────────────────────────

def classify_crop(crop_bgr: np.ndarray, model: nn.Module, transform) -> float:
    """Return the positive-class probability for a single BGR crop."""
    from PIL import Image as _Image
    rgb = crop_bgr[:, :, ::-1]  # BGR → RGB
    pil = _Image.fromarray(rgb)
    tensor = transform(pil).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        if isinstance(logits, tuple):
            logits = logits[0]
        probs = torch.softmax(logits, dim=1)
    return float(probs[0, 1])  # class 1 = positive


def ensemble_predict_frame(
    frame_bgr: np.ndarray,
    yolo: YOLO,
    classifiers: list[tuple[nn.Module, transforms.Compose, str]],
    yolo_imgsz: int = 640,
    yolo_conf: float = 0.25,
    threshold: float = 0.5,
):
    """
    Run ensemble inference on a single BGR frame.

    Returns a list of dicts with keys:
        box_xyxy, yolo_conf, classifier_probs (dict), ensemble_score, kept
    """
    import cv2

    yolo_results = yolo.predict(source=frame_bgr, imgsz=yolo_imgsz, conf=yolo_conf, verbose=False)
    detections = []
    for result in yolo_results:
        if result.boxes is None or len(result.boxes) == 0:
            continue
        for i in range(len(result.boxes)):
            x1, y1, x2, y2 = result.boxes.xyxy[i].cpu().numpy().astype(int)
            conf = float(result.boxes.conf[i].cpu())
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_bgr.shape[1], x2), min(frame_bgr.shape[0], y2)
            if x2 - x1 < 5 or y2 - y1 < 5:
                continue
            crop = frame_bgr[y1:y2, x1:x2]

            cls_probs = {}
            for model, tfm, name in classifiers:
                cls_probs[name] = classify_crop(crop, model, tfm)

            all_probs = [conf] + list(cls_probs.values())
            ensemble_score = float(np.mean(all_probs))

            detections.append({
                "box_xyxy": (int(x1), int(y1), int(x2), int(y2)),
                "yolo_conf": conf,
                "classifier_probs": cls_probs,
                "ensemble_score": ensemble_score,
                "kept": ensemble_score >= threshold,
            })
    return detections


# ── Register the classifier list for downstream cells ──────────────────
# Each entry: (model, val_transform, display_name)
ensemble_classifiers = [
    (alexnet_model, classifier_val_transform, "alexnet"),
    (inception_model, inception_val_transform, "inception"),
]

print(f"Ensemble ready — YOLO + {[name for _, _, name in ensemble_classifiers]}")
print(f"Ensemble threshold: {ensemble_threshold}")

## Step 5: Get center pixel coordinates (ensemble)
1. Pick an image or folder to test the ensemble detector.
2. YOLO proposes boxes; AlexNet and Inception vote on each crop.
3. Only detections exceeding `ensemble_threshold` are reported.

In [ ]:
import cv2

# Use your latest trained YOLO weights for inference
inference_weights_path = find_most_recent_weights(runs_detect_dir) or starting_model
inference_weights_path = Path(inference_weights_path)
if not inference_weights_path.exists():
    raise FileNotFoundError(f"Inference weights not found: {inference_weights_path}")

try:
    inference_yolo = YOLO(str(inference_weights_path))
except Exception as exc:
    raise RuntimeError(f"Failed to load YOLO from {inference_weights_path}: {exc}") from exc

test_image_path = Path("D:\\BR Footage\\Gharial\\target\\validate\\01_05_2024 54")
if not test_image_path.exists():
    raise FileNotFoundError(f"Inference source path does not exist: {test_image_path}")

# Collect all image paths
if test_image_path.is_dir():
    test_images = sorted(
        p for p in test_image_path.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
else:
    test_images = [test_image_path]

classifier_names = [name for _, _, name in ensemble_classifiers]

detected_centers = []
for img_path in test_images:
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue
    detections = ensemble_predict_frame(
        frame,
        inference_yolo,
        ensemble_classifiers,
        yolo_conf=0.25,
        threshold=ensemble_threshold,
    )
    for det in detections:
        if not det["kept"]:
            continue
        x1, y1, x2, y2 = det["box_xyxy"]
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        detected_centers.append({
            "file": img_path.name,
            "center_x": float(cx),
            "center_y": float(cy),
            "ensemble_score": det["ensemble_score"],
            "yolo_conf": det["yolo_conf"],
            **{name: det["classifier_probs"][name] for name in classifier_names},
        })

print("Ensemble centers (x, y):")
if not detected_centers:
    print("No ensemble-confirmed detections found.")
else:
    # Header
    model_cols = "  ".join(f"{name:>10}" for name in classifier_names)
    print(f"  {'file':<30}  {'center_x':>9}  {'center_y':>9}  {'ensemble':>9}  {'yolo':>9}  {model_cols}")
    print("  " + "-" * (30 + 12 * 2 + 12 * 2 + 12 * len(classifier_names)))
    for c in detected_centers:
        model_vals = "  ".join(f"{c[name]:>10.3f}" for name in classifier_names)
        print(f"  {c['file']:<30}  {c['center_x']:>9.1f}  {c['center_y']:>9.1f}  "
              f"{c['ensemble_score']:>9.3f}  {c['yolo_conf']:>9.3f}  {model_vals}")

WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\core\src\alloc.cpp:73: error: (-4:Insufficient memory) Failed to allocate 6220800 bytes in function 'cv::OutOfMemoryError'


## Step 6: Measure distance traveled (ensemble)
1. Set a folder of ordered frames and (optionally) a CSV output path.
2. Run ensemble detection across frames to compute center points and distances.
3. Review the printed table or saved CSV for totals and per-frame stats.

In [ ]:
# Folder of frames in order (e.g., a *_frames folder or your val/train folder)
frames_root = Path("D:\\BR Footage\\Gharial\\target\\validate\\master_frames")
output_csv_path = None
confidence_threshold = 0.00
pixels_per_unit_distance = None

if not frames_root.exists() or not frames_root.is_dir():
    raise FileNotFoundError(f"frames_root does not exist or is not a directory: {frames_root}")
if confidence_threshold < 0 or confidence_threshold > 1:
    raise ValueError(f"confidence_threshold must be in [0, 1], got {confidence_threshold}")
if pixels_per_unit_distance is not None and float(pixels_per_unit_distance) <= 0:
    raise ValueError("pixels_per_unit_distance must be > 0 when provided.")

# Ensure we have a YOLO model for inference
if "inference_yolo" not in globals():
    inference_weights_path = find_most_recent_weights(runs_detect_dir) or starting_model
    inference_weights_path = Path(inference_weights_path)
    if not inference_weights_path.exists():
        raise FileNotFoundError(f"Inference weights not found: {inference_weights_path}")
    inference_yolo = YOLO(str(inference_weights_path))

classifier_names = [name for _, _, name in ensemble_classifiers]


def detect_best_center_ensemble(frame_path: Path) -> dict | None:
    """Detect the highest-scoring ensemble detection in a single frame."""
    frame = cv2.imread(str(frame_path))
    if frame is None:
        return None

    detections = ensemble_predict_frame(
        frame,
        inference_yolo,
        ensemble_classifiers,
        yolo_conf=confidence_threshold,
        threshold=ensemble_threshold,
    )

    kept = [d for d in detections if d["kept"]]
    if not kept:
        return None

    best = max(kept, key=lambda d: d["ensemble_score"])
    x1, y1, x2, y2 = best["box_xyxy"]
    return {
        "center_x": (x1 + x2) / 2.0,
        "center_y": (y1 + y2) / 2.0,
        "ensemble_score": best["ensemble_score"],
        "yolo_conf": best["yolo_conf"],
        **{name: best["classifier_probs"][name] for name in classifier_names},
    }


frame_image_paths = sorted(
    [p for p in frames_root.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
)
if not frame_image_paths:
    raise RuntimeError("No frames found. Check frames_root.")

table_rows = []
previous_center = None
cumulative_distance = 0.0
for index, frame_path in enumerate(frame_image_paths):
    center = detect_best_center_ensemble(frame_path)
    if center is None:
        row = {
            "frame": frame_path.name,
            "frame_index": index,
            "center_x": None,
            "center_y": None,
            "ensemble_score": None,
            "yolo_conf": None,
        }
        for name in classifier_names:
            row[name] = None
        row["distance_from_prev"] = None
        row["cumulative_distance"] = cumulative_distance
        table_rows.append(row)
        continue

    distance_from_previous = None
    if previous_center is not None:
        delta_x = center["center_x"] - previous_center["center_x"]
        delta_y = center["center_y"] - previous_center["center_y"]
        distance_from_previous = float((delta_x ** 2 + delta_y ** 2) ** 0.5)
        if pixels_per_unit_distance:
            distance_from_previous /= float(pixels_per_unit_distance)
        cumulative_distance += distance_from_previous

    row = {
        "frame": frame_path.name,
        "frame_index": index,
        "center_x": center["center_x"],
        "center_y": center["center_y"],
        "ensemble_score": center["ensemble_score"],
        "yolo_conf": center["yolo_conf"],
    }
    for name in classifier_names:
        row[name] = center[name]
    row["distance_from_prev"] = distance_from_previous
    row["cumulative_distance"] = cumulative_distance
    table_rows.append(row)
    previous_center = center

if output_csv_path:
    output_csv_path = Path(output_csv_path)
    output_csv_path.parent.mkdir(parents=True, exist_ok=True)

if output_csv_path:
    with open(output_csv_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=table_rows[0].keys())
        writer.writeheader()
        writer.writerows(table_rows)
    print(f"Saved CSV to {output_csv_path}")

# Pretty-print with per-model confidences
model_cols_hdr = "  ".join(f"{name:>10}" for name in classifier_names)
print(f"{'frame':<30}  {'cx':>7}  {'cy':>7}  {'ens':>7}  {'yolo':>7}  {model_cols_hdr}  {'dist':>8}  {'cumul':>8}")
print("-" * (30 + 10 * 2 + 10 * 2 + 10 * len(classifier_names) + 10 * 2 + 6))
for row in table_rows[:10]:
    if row["center_x"] is None:
        model_vals = "  ".join(f"{'—':>10}" for _ in classifier_names)
        print(f"{row['frame']:<30}  {'—':>7}  {'—':>7}  {'—':>7}  {'—':>7}  {model_vals}  {'—':>8}  {row['cumulative_distance']:>8.1f}")
    else:
        model_vals = "  ".join(f"{row[name]:>10.3f}" for name in classifier_names)
        dist_str = f"{row['distance_from_prev']:>8.1f}" if row["distance_from_prev"] is not None else f"{'—':>8}"
        print(f"{row['frame']:<30}  {row['center_x']:>7.1f}  {row['center_y']:>7.1f}  "
              f"{row['ensemble_score']:>7.3f}  {row['yolo_conf']:>7.3f}  {model_vals}  {dist_str}  {row['cumulative_distance']:>8.1f}")

print(f"\nTotal frames: {len(table_rows)}  (showing first 10)")

df = None

,frame,frame_index,center_x,center_y,confidence,distance_from_prev,cumulative_distance
0,01050110_0.000.png,0,49.902660,726.199463,0.007852,NaN,0.000000e+00
1,01050110_0.000_04e59a6a-0add-4c90-825f-c5343de...,1,49.902660,726.199463,0.007852,0.000000,0.000000e+00
2,frame_0001.jpg,2,1753.467163,811.859741,0.031569,1705.716770,1.705717e+03
3,frame_0001_01dd8131-efcc-491a-a755-a12c5da8a8b...,3,219.768311,107.666611,0.000558,1687.637442,3.393354e+03
4,frame_0001_021410a9-4f49-4788-a74a-640f7160b9a...,4,428.960083,819.111328,0.328727,741.562394,4.134917e+03
...,...,...,...,...,...,...,...
1301,frame_0002_fe0c541d-e87f-4393-bcf8-32e8110cb6e...,1301,1711.934448,865.341614,0.000628,1665.675893,1.025593e+06
1302,frame_0002_fe2ce13d-fd7e-4df1-a393-9e782545dcb...,1302,1575.779785,848.991821,0.756266,137.132812,1.025730e+06
1303,frame_0002_ff65b8a3-f83c-4339-87d1-7afd44f1037...,1303,1580.717285,956.232910,0.020209,107.354693,1.025837e+06
1304,frame_0002_fff4e5ad-18d4-4756-9082-598061b42b6...,1304,647.372314,793.983032,0.626855,947.342524,1.026785e+06


In [ ]:
df_obj = globals().get("df", None)
if df_obj is not None:
    pace_distance_threshold = 3
    df_obj["pacing_flag"] = df_obj["distance_from_prev"].apply(
        lambda distance: 1 if distance is not None and distance > pace_distance_threshold else 0
    )
    df = df_obj
else:
    print("No pandas DataFrame available; skipping pacing flag cell.")

In [ ]:
df_obj = globals().get("df", None)
if df_obj is not None:
    df_obj[df_obj["center_x"].notnull()].sort_values("confidence", ascending=False)
else:
    print("No pandas DataFrame available; skipping confidence sort view.")

,frame,frame_index,center_x,center_y,confidence,distance_from_prev,cumulative_distance,pacing
394,frame_0001_95e073ab-8929-4f06-8995-e1d12a38f6c...,394,629.829529,869.335876,0.846734,264.564696,301208.466981,1
675,frame_0002_089a1ebe-b207-436b-92b1-eb1bf45b215...,675,633.563782,866.608521,0.842240,594.719794,522943.760246,1
1077,frame_0002_ab27a6df-d294-414e-b4b6-c5f086c8a10...,1077,345.435608,900.640381,0.837746,131.522390,856588.495269,1
104,frame_0001_254c9716-93a0-4e0e-b987-f18216e958f...,104,341.265839,896.333130,0.836498,150.809839,77916.036737,1
282,frame_0001_6c59fa6c-d501-4190-8332-f75672b5789...,282,1586.203369,828.547485,0.834199,1507.665054,214226.602406,1
...,...,...,...,...,...,...,...,...
804,frame_0002_3df61b9c-9507-4491-8032-1b87ebb521f...,804,53.296761,1064.193237,0.000040,439.432102,630533.421939,1
984,frame_0002_8aeeefe2-f653-402c-8277-fd327e7ec35...,984,1733.834717,134.836731,0.000039,1452.820883,779594.062456,1
283,frame_0001_6cad9370-551a-47c5-89fc-c25579199e2...,283,1460.843018,739.094238,0.000037,154.003575,214380.605981,1
402,frame_0001_9a1be409-748a-420d-964c-b5202aeb420...,402,46.218723,76.292770,0.000036,1277.345457,309507.497502,1
